# Tutorial 10: What Are the Chances?

**Programming Design Principles / Maths for IT**

Probability is the mathematics of uncertainty. It gives us a precise language for talking about how likely things are, and it underpins everything from weather forecasts to medical diagnosis to machine learning. Today we build the foundations, and we will use an approach that is unique to programming: we can *simulate* random events to verify our calculations.

## Basic Probability

The probability of an event is a number between 0 (impossible) and 1 (certain). When all outcomes are equally likely, the probability of an event A is:

$$P(A) = \frac{\text{number of favourable outcomes}}{\text{total number of outcomes}}$$

Flip a fair coin: there are 2 equally likely outcomes and 1 favourable outcome (heads), so $P(\text{heads}) = \frac{1}{2} = 0.5$.

Roll a fair die: $P(\text{rolling a 4}) = \frac{1}{6}$. $P(\text{rolling an even number}) = \frac{3}{6} = \frac{1}{2}$.

### Your turn

Write a function `probability(favourable, total)` that computes a basic probability. Include a docstring and think about edge cases: what if total is 0? What if favourable is greater than total?

In [1]:
def probability(favourable, total):
    """Return the probability of an event with `favourable` outcomes out of `total` equally likely outcomes.

    If total is 0 there is nothing to divide by, so we return 0.0 rather than raising an error.
    Favourable cannot exceed total for a genuine probability, so we cap the result at 1.0.
    """
    if total == 0:
        return 0.0
    if favourable > total:
        return 1.0
    return favourable / total


In [2]:
print("P(specific die face):", probability(1, 6))
print("P(heart):", probability(13, 52))
print("P(ace):", probability(4, 52))


P(specific die face): 0.16666666666666666
P(heart): 0.25
P(ace): 0.07692307692307693


## Compound Events

Things get interesting when we combine events. There are a few key rules.

**The complement rule**: the probability that an event does *not* happen is $1 - P(A)$. If there is a 30% chance of rain, there is a 70% chance of no rain.

**The addition rule**: for two events that cannot both happen at the same time (*mutually exclusive* events):

$$P(A \text{ or } B) = P(A) + P(B)$$

Rolling a 2 or a 5 on a die: $P = \frac{1}{6} + \frac{1}{6} = \frac{2}{6} = \frac{1}{3}$

**The general addition rule**: when events *can* overlap:

$$P(A \text{ or } B) = P(A) + P(B) - P(A \text{ and } B)$$

We subtract the overlap to avoid counting it twice.

In [3]:
# Card example: probability of drawing an Ace OR a Heart
p_ace = 4 / 52
p_heart = 13 / 52
p_ace_of_hearts = 1 / 52    # the overlap: it's both an ace AND a heart

p_ace_or_heart = p_ace + p_heart - p_ace_of_hearts
print("P(Ace or Heart):", p_ace_or_heart)
print("That's", round(p_ace_or_heart, 4), "or about", round(p_ace_or_heart * 100, 1), "%")

P(Ace or Heart): 0.3076923076923077
That's 0.3077 or about 30.8 %


**The multiplication rule**: for *independent* events (one happening does not affect the other):

$$P(A \text{ and } B) = P(A) \times P(B)$$

Flipping heads twice in a row: $P = \frac{1}{2} \times \frac{1}{2} = \frac{1}{4}$

When events are *not* independent (like drawing cards without replacement), the second probability depends on the first:

$$P(\text{two aces in a row}) = \frac{4}{52} \times \frac{3}{51}$$

After drawing one ace, there are 3 aces left among 51 remaining cards.

### Your turn

Using your `probability` function and the combination functions from Tutorial 9, work through these card probability questions. Write your reasoning before computing.

1. What is the probability of drawing a face card (Jack, Queen, or King)?
2. What is the probability of drawing a card that is red *and* a face card?
3. What is the probability of drawing a card that is red *or* a face card?
4. If you draw two cards without replacement, what is the probability both are hearts?
5. What is the probability of being dealt a royal flush (A, K, Q, J, 10 all of the same suit) in a 5-card hand?

Cell 9 below reaches for the combination-counting tools from Tutorial 9. Since each
tutorial is its own notebook, we bring `factorial` and `combinations` forward rather
than assuming they are still sitting in memory.

In [4]:
def factorial(n):
    """Return n! (n factorial), carried forward from Tutorial 9."""
    if n == 0:
        return 1
    product = 1
    for i in range(1, n + 1):
        product = product * i
    return product


def combinations(n, r):
    """Return C(n, r), carried forward from Tutorial 9."""
    if r > n:
        return 0
    return factorial(n) // (factorial(r) * factorial(n - r))


In [5]:
# 1. Face cards: 12 face cards (J, Q, K in each of 4 suits) out of 52
p_face = probability(12, 52)
print("1. P(face card):", round(p_face, 4))

# 2. Red AND face card: 6 red face cards (J, Q, K of hearts and diamonds) out of 52
p_red_and_face = probability(6, 52)
print("2. P(red and face card):", round(p_red_and_face, 4))

# 3. Red OR face card (careful: these overlap!)
p_red = probability(26, 52)
p_red_or_face = p_red + p_face - p_red_and_face
print("3. P(red or face card):", round(p_red_or_face, 4))

# 4. Two hearts without replacement
p_two_hearts = probability(13, 52) * probability(12, 51)
print("4. P(two hearts, no replacement):", round(p_two_hearts, 4))

# 5. Royal flush (hint: how many royal flushes are possible?
#    how many total 5-card hands are possible?)
royal_flushes = 4                       # one per suit
total_hands = combinations(52, 5)
p_royal_flush = probability(royal_flushes, total_hands)
print("5. P(royal flush):", p_royal_flush)


1. P(face card): 0.2308
2. P(red and face card): 0.1154
3. P(red or face card): 0.6154
4. P(two hearts, no replacement): 0.0588
5. P(royal flush): 1.5390771693292702e-06


## Simulation: Testing Probability with Code

One of the wonderful things about having programming skills: we can *simulate* random events to verify our calculations. If we flip a simulated coin 10,000 times, we should see heads about 50% of the time.

Python's `random` module provides the tools:

In [6]:
import random

# Simulate flipping a coin 10,000 times
num_flips = 10000
heads_count = 0

for i in range(num_flips):
    flip = random.choice(["heads", "tails"])
    if flip == "heads":
        heads_count = heads_count + 1

proportion = heads_count / num_flips
print("Heads:", heads_count, "out of", num_flips)
print("Proportion:", round(proportion, 4))
print("Expected:   0.5")

Heads: 5039 out of 10000
Proportion: 0.5039
Expected:   0.5


The simulated proportion will not be exactly 0.5, but it should be close. The more trials we run, the closer it gets. This is the *law of large numbers* in action.

### Your turn

Write a function `simulate_coin_flips(num_trials)` that returns the proportion of heads. Then call it with 100, 1000, 10000, and 100000 trials. What do you notice about the proportion as the number of trials increases?

In [7]:
def simulate_coin_flips(num_trials):
    """Return the proportion of heads in `num_trials` simulated fair coin flips."""
    heads_count = 0
    for i in range(num_trials):
        flip = random.choice(["heads", "tails"])
        if flip == "heads":
            heads_count = heads_count + 1
    return heads_count / num_trials


In [8]:
for trials in [100, 1000, 10000, 100000]:
    print(trials, "trials -> proportion of heads:", round(simulate_coin_flips(trials), 4))
# As the number of trials grows, the proportion settles in closer to 0.5 --
# the law of large numbers again.


100 trials -> proportion of heads: 0.44
1000 trials -> proportion of heads: 0.496
10000 trials -> proportion of heads: 0.4915
100000 trials -> proportion of heads: 0.4988


### Simulating card draws

Let's simulate the card probabilities we calculated earlier. We will represent a deck of cards and draw from it:

In [9]:
def make_deck():
    """Create a standard 52-card deck as a list of (rank, suit) tuples."""
    ranks = ["2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K", "A"]
    suits = ["Hearts", "Diamonds", "Clubs", "Spades"]
    deck = []
    for suit in suits:
        for rank in ranks:
            deck.append((rank, suit))
    return deck

deck = make_deck()
print("Deck size:", len(deck))
print("First 5 cards:", deck[:5])
print("Last 5 cards:", deck[-5:])

Deck size: 52
First 5 cards: [('2', 'Hearts'), ('3', 'Hearts'), ('4', 'Hearts'), ('5', 'Hearts'), ('6', 'Hearts')]
Last 5 cards: [('10', 'Spades'), ('J', 'Spades'), ('Q', 'Spades'), ('K', 'Spades'), ('A', 'Spades')]


### Your turn

Write a function `simulate_draw(num_trials)` that simulates drawing a single card from a shuffled deck many times, and counts how often you get an Ace or a Heart. Compare the simulated proportion to your calculated probability from earlier.

Hint: `random.shuffle(deck)` shuffles a list in place, and `deck[0]` gives the top card.

In [10]:
def simulate_draw(num_trials):
    """Simulate drawing a single card `num_trials` times and return the proportion that are an Ace or a Heart."""
    deck = make_deck()
    ace_or_heart_count = 0
    for i in range(num_trials):
        random.shuffle(deck)
        card = deck[0]
        rank, suit = card
        if rank == "A" or suit == "Hearts":
            ace_or_heart_count = ace_or_heart_count + 1
    return ace_or_heart_count / num_trials


In [11]:
simulated = simulate_draw(20000)
calculated = p_ace_or_heart
print("Simulated P(ace or heart):", round(simulated, 4))
print("Calculated P(ace or heart):", round(calculated, 4))


Simulated P(ace or heart): 0.315
Calculated P(ace or heart): 0.3077


### A more complex simulation

Write a function that simulates drawing two cards without replacement, and counts how often both are hearts. Compare to your calculation.

Then, if you are feeling ambitious, simulate dealing 5-card hands and count how many are a royal flush. You will need a *lot* of trials (millions) to see even one, which gives you an intuitive sense of just how rare they are.

In [12]:
def simulate_two_hearts(num_trials):
    """Simulate drawing two cards without replacement `num_trials` times and return the proportion where both are hearts."""
    deck = make_deck()
    both_hearts_count = 0
    for i in range(num_trials):
        random.shuffle(deck)
        first, second = deck[0], deck[1]
        if first[1] == "Hearts" and second[1] == "Hearts":
            both_hearts_count = both_hearts_count + 1
    return both_hearts_count / num_trials

simulated_two_hearts = simulate_two_hearts(20000)
print("Simulated P(two hearts):", round(simulated_two_hearts, 4))
print("Calculated P(two hearts):", round(p_two_hearts, 4))


Simulated P(two hearts): 0.0583
Calculated P(two hearts): 0.0588


In [13]:
def simulate_royal_flush(num_trials):
    """Simulate dealing 5-card hands `num_trials` times and return the proportion that are a royal flush."""
    deck = make_deck()
    royal_ranks = {"10", "J", "Q", "K", "A"}
    royal_flush_count = 0
    for i in range(num_trials):
        random.shuffle(deck)
        hand = deck[:5]
        ranks = set(card[0] for card in hand)
        suits = set(card[1] for card in hand)
        if ranks == royal_ranks and len(suits) == 1:
            royal_flush_count = royal_flush_count + 1
    return royal_flush_count / num_trials

# A royal flush is about a 1-in-650,000 hand, so even a few hundred thousand
# trials will usually show zero -- that scarcity is the point of the exercise.
trials = 200000
proportion = simulate_royal_flush(trials)
print("Simulated over", trials, "trials:", proportion)
print("Calculated:", p_royal_flush)


Simulated over 200000 trials: 0.0
Calculated: 1.5390771693292702e-06


## Conditional Probability

Sometimes the probability of an event depends on what has already happened. The probability of drawing a heart *given that* we already drew a heart (without replacement) is $\frac{12}{51}$, not $\frac{13}{52}$.

This is called *conditional probability* and is written $P(B|A)$ -- "the probability of B given A":

$$P(B|A) = \frac{P(A \text{ and } B)}{P(A)}$$

We will not go deep into this today, but it is worth knowing the concept because it is foundational in machine learning (where Bayes' theorem, which builds on conditional probability, is everywhere).

### Your turn

If you draw one card and see that it is red, what is the probability that it is a heart? Think about this intuitively first, then verify with the formula.

In [14]:
# Intuition first: half the deck is red (26 cards), and all 13 hearts are red,
# so among red cards, hearts make up 13 out of 26 -- exactly one half.
p_red = probability(26, 52)
p_heart_and_red = probability(13, 52)   # every heart is red
p_heart_given_red = p_heart_and_red / p_red
print("P(heart | red):", p_heart_given_red)


P(heart | red): 0.5


## Reflection

We have covered the fundamental rules of probability (complement, addition, multiplication) and learned to use simulation to verify our calculations. The simulation approach is not just a teaching tool -- it is a genuine technique used in industry when problems become too complex for exact calculation.

The combination of mathematical reasoning and computational verification is powerful: we calculate an expected probability, then simulate to check. If they agree, we have confidence in both. If they disagree, we have a bug to find -- and finding bugs is learning.

What was most surprising about the relationship between calculation and simulation?

